In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import requests
from tqdm import tqdm
tqdm.pandas()

In [ ]:
pd.set_option('display.max_colwidth', None)
df = pd.read_csv('/Users/giorgiotamo/Desktop/GT/Programming/Lavoro/GitHub/JobSeekerAgency/output/bnf_projects_relevant.csv')
# df[['name_en','city','relevant']]

geolocator = Nominatim(user_agent="bnf_distance", timeout=10)

# Geocode Biel once
biel_loc = geolocator.geocode("Biel, Switzerland")
biel_coords = (biel_loc.latitude, biel_loc.longitude)

def get_driving_distance(city):
    """Get driving distance from Biel to city via OSRM."""
    loc = geolocator.geocode(f"{city}, Switzerland")
    if not loc:
        return None
    url = (
        f"http://router.project-osrm.org/route/v1/driving/"
        f"{biel_coords[1]},{biel_coords[0]};{loc.longitude},{loc.latitude}"
        f"?overview=false"
    )
    r = requests.get(url, timeout=10).json()
    if r.get("routes"):
        return round(r["routes"][0]["distance"] / 1000, 1)
    return round(geodesic(biel_coords, (loc.latitude, loc.longitude)).km, 1)

# Compute distances only for unique cities, then map back
unique_cities = df['city'].dropna().unique()
print(f"Computing distances for {len(unique_cities)} unique cities...")
city_distances = {city: get_driving_distance(city.strip()) for city in tqdm(unique_cities)}

df['distance_to_biel_km'] = df['city'].map(city_distances)
df[['name_en', 'city', 'relevant', 'distance_to_biel_km']].sort_values('distance_to_biel_km')

Computing distances for 22 unique cities...


100%|██████████| 22/22 [00:27<00:00,  1.27s/it]


,name_en,city,relevant,distance_to_biel_km
3,AI Powered Human Resource Application,Lyss,1,12.4
24,"Edge AI for environmental monitoring, with a focus on bioacoustics",Zollikofen,1,30.7
21,Development of machine-learning identification tools for soil biodiversity (protists and/or invertebrates),Neuchâtel,1,31.6
23,"Conducting and evaluating veterinary epidemiological studies, contributing to epidemiological-statistical services (project consulting) as well as to the preparation of risk analyses",Liebefeld,1,45.0
33,Live Media Fact Checking and Analysis Using Machine Learning,Yverdon-les-Bains,1,70.0
34,Medical Insurance Anomaly Detection for Healthcare Cost and Quality Improvement,Yverdon-les-Bains,1,70.0
49,Spatial Crisis Zone Identification Using Multimodal Map Data,Yverdon-les-Bains,1,70.0
55,Validation of R packages for clinical research,Basel,1,91.6
53,Structural Biology of Antibiotics Production & Action and Antimicrobial Resistance,Basel,1,91.6
35,Drug development for worm diseases,Allschwil,1,94.3


In [ ]:
def compareVersion(version1: str, version2: str) -> int:
    v1_parts = list(map(int, version1.split('.')))
    v2_parts = list(map(int, version2.split('.')))
    
    max_length = max(len(v1_parts), len(v2_parts))
    v1_parts.extend([0] * (max_length - len(v1_parts)))
    v2_parts.extend([0] * (max_length - len(v2_parts)))

    print(v1_parts, v2_parts)
    
    for part1, part2 in zip(v1_parts, v2_parts):
        if part1 > part2:
            return -1  # swapped
        elif part1 < part2:
            return 1   # swapped
    return 0


assert compareVersion ("1.0.0", "1.0.1") == 1
assert compareVersion ("1", "1.0.0") == 0
assert compareVersion("2.0.1", "2.0.10") == 1
assert compareVersion ("1.0.0", "2.0.0.13") == 1
assert compareVersion ("18.0", "0.0.0.4")== -1
assert compareVersion("1.0.1.23", "1.0.11.1") == 1
assert compareVersion("1.0.1.23", "1.0") == -1
assert compareVersion ("1.0.0", "1.0") == 0
assert compareVersion ("1.0.0.0", "1.0") == 0
assert compareVersion("1.0.0.2", "1.0") == -1
assert compareVersion("1.10.0.2", "1.2.0") == -1

[1, 0, 0] [1, 0, 1]
[1, 0, 0] [1, 0, 0]
[2, 0, 1] [2, 0, 10]
[1, 0, 0, 0] [2, 0, 0, 13]
[18, 0, 0, 0] [0, 0, 0, 4]
[1, 0, 1, 23] [1, 0, 11, 1]
[1, 0, 1, 23] [1, 0, 0, 0]
[1, 0, 0] [1, 0, 0]
[1, 0, 0, 0] [1, 0, 0, 0]
[1, 0, 0, 2] [1, 0, 0, 0]
[1, 10, 0, 2] [1, 2, 0, 0]
